# Topic: XSS and CSRF Defense Architecture - Anti-CSRF token verification, same-origin checks, and SameSite cookie properties
 
## 1. CROSS-SITE REQUEST FORGERY (CSRF)
- **CSRF**: Occurs when a malicious website tricks a victim's browser into sending an HTTP request (like a bank transfer) to a target site where the victim is already authenticated.
  - Since browsers automatically append session cookies (cookies are sent with all requests to the domain that set them), the target site treats the request as authenticated, executing the malicious action.
 
## 2. DEFENSIVE ARCHITECTURE
- **Anti-CSRF Tokens (Synchronizer Token Pattern)**:
  - The server generates a cryptographically secure random token and binds it to the user's active session.
  - The token is placed inside all writeable form templates (e.g. `<input type="hidden" name="csrf_token" value="abc...">`).
  - When the client submits the form (POST), the server compares the submitted form token with the token stored in the user's session.
  - Since cross-origin scripts cannot read the token value (due to the Same-Origin Policy), attackers cannot construct valid forms, blocking the forge attack.
- **SameSite Cookie Attribute**:
  - Setting `SameSite=Lax` or `SameSite=Strict` instruct browsers not to send cookies along with cross-origin requests.
 
---


In [ ]:
import secrets

class CSRFProtectionSystem:
    """Simulates a server-side session and anti-CSRF token validator."""
    def __init__(self):
        # Session storage: session_id -> {username, csrf_token}
        self.session_store = {}

    def login_user(self, username):
        """Authenticates user, creates session, and generates a secure CSRF token."""
        session_id = secrets.token_hex(16)
        # Generate cryptographically secure CSRF token bound to this session
        csrf_token = secrets.token_hex(32)
        
        self.session_store[session_id] = {
            "username": username,
            "csrf_token": csrf_token
        }
        return session_id, csrf_token

    def execute_transfer(self, session_id, amount, recipient, form_csrf_token=None):
        """Processes balance transfers, validating the CSRF token."""
        # 1. Validate Session Cookie
        session = self.session_store.get(session_id)
        if not session:
            return "401 Unauthorized: Invalid session ID."

        # 2. Validate Anti-CSRF Token
        expected_token = session["csrf_token"]
        if form_csrf_token is None:
            # Missing token indicating potential forged cross-site POST attempt
            return "403 Forbidden: Missing Anti-CSRF Token! Transfer blocked."
            
        # Constant-time comparison to prevent timing attacks
        if not secrets.compare_digest(expected_token, form_csrf_token):
            return "403 Forbidden: Invalid Anti-CSRF Token! Transfer blocked."
            
        return f"200 OK: Successfully transferred ${amount} to {recipient} from user '{session['username']}'."



In [ ]:
# Testing the CSRF Protection System
print("--- Initializing CSRF Shield ---")
shield = CSRFProtectionSystem()

# User 'alice' logs in
session_cookie, alice_csrf_token = shield.login_user("alice")
print(f"Alice's Session Cookie:  {session_cookie}")
print(f"Alice's Anti-CSRF Token: {alice_csrf_token}")



In [ ]:
# Scenario A: Valid Transfer from Alice (includes valid form token)
print("\n--- Valid Transfer Attempt ---")
response_valid = shield.execute_transfer(
    session_id=session_cookie,
    amount=100.0,
    recipient="bob",
    form_csrf_token=alice_csrf_token
)
print(response_valid)



In [ ]:
# Scenario B: CSRF Forge Attack Attempt
# Attacker triggers a cross-origin form POST. The victim's browser appends the session cookie
# automatically, but the attacker does not know the csrf_token value!
print("\n--- Forged CSRF Attack Attempt (No token) ---")
response_forged = shield.execute_transfer(
    session_id=session_cookie,
    amount=1000.0,
    recipient="attacker",
    form_csrf_token=None  # Attacker cannot read/generate Alice's token
)
print(response_forged)
# Expected: Blocked with 403 Forbidden!
